# Lab 6.1 · Gephi Workshop — From Graph to Figure
**Chapter 6 · Visual Network Analytics with Gephi and Project Kick-off**

**Goal:** rebuild your Chapter 4 and 5 results in Gephi and produce **one captioned, publication-ready network figure**.
**Data:** `nx.karate_club_graph()` · `data/ev_mentions.csv` + `data/ev_users.csv` (Lab 4.1) · `data/ev_cascades.csv` (Lab 5.1) — all synthetic.

This notebook only **prepares** the files. The real work happens in **Gephi** (desktop, version 0.10.1 — Java is included in the installer).
No Python at hand? Ready-made files are in [`data/gephi/`](../data/gephi/).

📖 Interactive study guide: [Chapter 6](https://mehrdadjalali-ai.github.io/SocialMediaAnayltics/interactive/Chapter6_Gephi_Project.html) — a mini-Gephi playground with the same karate numbers.

| File written to `outputs/` | Open it in Gephi for |
|---|---|
| `karate_ch6.gexf` | Steps 1–7: check, layout, statistics vs. Chapter 4, colour, size, filters |
| `karate_cascade.gexf` | Step 8: replay one Independent Cascade run on the Timeline |
| `ev_mentions_ch6.gexf` | Challenge 2: your EV network, coloured by mean sentiment |
| `ev_cascade_c01.gexf`, `ev_cascade_c05.gexf` | Challenge 3: a broadcast vs. a viral cascade on the Timeline |

In [1]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, OUTD = ROOT / "data", ROOT / "outputs"
OUTD.mkdir(exist_ok=True)
SEED = 42
import random
import pandas as pd, networkx as nx

## Walkthrough 1 · The karate club, ready for Gephi (Steps 1–7)
We store the Chapter 4 results as **node attributes**, so you can compare them with Gephi’s own statistics in the Data Laboratory.

In [2]:
G = nx.karate_club_graph()                      # edge weights + "club" attribute
comms = nx.community.louvain_communities(G, seed=SEED)
for i, c in enumerate(comms):
    for n in c:
        G.nodes[n]["community_nx"] = i
pr = nx.pagerank(G)                             # uses the edge weights
for n in G:
    G.nodes[n]["pagerank_nx"] = round(pr[n], 4)

nx.write_gexf(G, OUTD / "karate_ch6.gexf")
print(G.number_of_nodes(), "nodes ·", G.number_of_edges(), "edges · directed:", G.is_directed())
print("communities:", len(comms), "· modularity:", round(nx.community.modularity(G, comms), 3))

34 nodes · 78 edges · directed: False
communities: 4 · modularity: 0.427


## Walkthrough 2 · One cascade for the Timeline (Step 8)
Exactly the Independent Cascade code from Chapter 5 — seed node 0, p = 0.25, `random.Random(1)`. The adoption step becomes the node’s `start` time, so Gephi’s **Timeline** can replay the spread.

In [3]:
H = nx.karate_club_graph()
rng = random.Random(1)
t = {0: 0}; frontier = [0]; step = 0
while frontier:
    step += 1; new = []
    for u in frontier:
        for v in H.neighbors(u):
            if v not in t and rng.random() < 0.25:
                t[v] = step; new.append(v)
    frontier = new

H.graph["mode"] = "dynamic"
for n in H:
    H.nodes[n]["adopted"] = "yes" if n in t else "no"
    if n in t:
        H.nodes[n]["start"] = float(t[n])
nx.write_gexf(H, OUTD / "karate_cascade.gexf")
print("adopters:", len(t), "of 34 · steps:", max(t.values()))
print("cumulative per step:", [sum(1 for s in t.values() if s <= k) for k in range(max(t.values()) + 1)])

adopters: 20 of 34 · steps: 5
cumulative per step: [1, 5, 9, 15, 17, 20]


## Walkthrough 3 · Your EV mention network (Challenge 2)
Everything Gephi should show must be an **attribute first**: user group and language (`ev_users.csv`), the **mean sentiment of a user’s mentions** (−1 … +1, weighted by the number of mentions), the Louvain community and PageRank. IDs are pseudonymised — never put user names into Gephi labels.

In [4]:
m = pd.read_csv(DATA / "ev_mentions.csv"); users = pd.read_csv(DATA / "ev_users.csv")
E = nx.from_pandas_edgelist(m, "source", "target", ["weight"], create_using=nx.DiGraph)
nx.set_node_attributes(E, users.set_index("user_id")["group"].to_dict(), "group")
nx.set_node_attributes(E, users.set_index("user_id")["lang"].to_dict(), "lang")

m["score"] = m["sentiment"].map({"positive": 1, "neutral": 0, "negative": -1})
mean_sent = (m["score"] * m["weight"]).groupby(m["source"]).sum() / m.groupby("source")["weight"].sum()
for n in E:
    E.nodes[n]["mean_sentiment"] = round(float(mean_sent.get(n, 0.0)), 3)

U = E.to_undirected()
ev_comms = sorted(nx.community.louvain_communities(U, seed=SEED), key=len, reverse=True)
for i, c in enumerate(ev_comms):
    for n in c:
        E.nodes[n]["community_nx"] = i
ev_pr = nx.pagerank(E)
for n in E:
    E.nodes[n]["pagerank_nx"] = round(ev_pr[n], 4)

nx.write_gexf(E, OUTD / "ev_mentions_ch6.gexf")
print(E.number_of_nodes(), "users ·", E.number_of_edges(), "mention edges · Louvain:", len(ev_comms),
      "communities · modularity", round(nx.community.modularity(U, ev_comms), 3))
summary = pd.DataFrame({"community": [E.nodes[n]["community_nx"] for n in E], "group": [E.nodes[n]["group"] for n in E],
                        "mean_sentiment": [E.nodes[n]["mean_sentiment"] for n in E]})
summary.groupby("community").agg(users=("group", "size"), main_group=("group", lambda s: s.value_counts().index[0]),
                                 share_main=("group", lambda s: round(s.value_counts(normalize=True).iloc[0], 2)),
                                 mean_sentiment=("mean_sentiment", lambda s: round(s.mean(), 2)))

260 users · 1208 mention edges · Louvain: 5 communities · modularity 0.443


,users,main_group,share_main,mean_sentiment
community,,,,
0,65,german_users,0.83,-0.15
1,63,critics,0.98,-0.42
2,49,news_brands,0.67,-0.05
3,46,owners,0.89,-0.07
4,37,owners,0.86,-0.01


## Walkthrough 4 · Two real cascades over time (Challenge 3)
`c01` (recall news) and `c05` (fire rumour) from Lab 5.1. Nodes are posts; edges go from parent to reply/repost; `start` is the number of **minutes after the first post** (timestamps are UTC).

In [5]:
casc = pd.read_csv(DATA / "ev_cascades.csv", parse_dates=["timestamp"])

def cascade_gexf(cid):
    g = casc[casc["cascade_id"] == cid].copy()
    t0 = g["timestamp"].min()
    C = nx.DiGraph(mode="dynamic")
    for r in g.itertuples():
        C.add_node(r.post_id, type=r.type, sentiment=r.sentiment,
                   start=round((r.timestamp - t0).total_seconds() / 60, 1))
    C.add_edges_from((p, q) for p, q in zip(g["parent_id"], g["post_id"]) if isinstance(p, str))
    root = g.loc[g["parent_id"].isna(), "post_id"].iloc[0]
    depth = nx.shortest_path_length(C, root)
    nx.set_node_attributes(C, depth, "depth")
    nx.write_gexf(C, OUTD / f"ev_cascade_{cid}.gexf")
    return {"cascade": cid, "topic": g["topic"].iloc[0], "posts": len(g), "depth": max(depth.values()),
            "duration_h": round((g["timestamp"].max() - t0).total_seconds() / 3600, 1),
            "share_negative": round((g["sentiment"] == "negative").mean(), 2)}

pd.DataFrame([cascade_gexf("c01"), cascade_gexf("c05")])

,cascade,topic,posts,depth,duration_h,share_negative
0,c01,recall news,212,4,2.6,0.23
1,c05,fire rumour,61,38,39.5,0.89


## Challenge 1 · Statistics vs. Chapter 4 — in Gephi
Open `karate_ch6.gexf` (File → Open → *New workspace*). Run **Average Degree, Graph Density, Network Diameter, Avg. Clustering Coefficient, Modularity, PageRank, Connected Components**. Fill in the worksheet: Gephi value · Python value · the Gephi settings you used. Run Modularity **three times** and report the range.

> 💡 **Hint:** Betweenness in the hundreds? Tick **Normalize**. PageRank of node 33 ≈ 0.101? Tick **Use edge weight**. Degree 48? You opened the *Weighted Degree* column.

The cell below prints the Python reference values — including the “wrong-setting” versions you may meet in Gephi.

In [6]:
btw_raw = nx.betweenness_centrality(G, normalized=False); btw = nx.betweenness_centrality(G)
pr_w, pr_u = nx.pagerank(G), nx.pagerank(G, weight=None)
cl = nx.clustering(G); avg_all = sum(cl.values()) / len(cl)
avg_skip = sum(v for n, v in cl.items() if G.degree(n) > 1) / sum(1 for n in G if G.degree(n) > 1)
ref = {
    "average degree": round(2 * G.number_of_edges() / G.number_of_nodes(), 3),
    "density": round(nx.density(G), 3),
    "diameter · avg. path length": (nx.diameter(G), round(nx.average_shortest_path_length(G), 2)),
    "betweenness of node 0 (normalised · raw)": (round(btw[0], 3), round(btw_raw[0], 2)),
    "PageRank of node 33 (weights · no weights)": (round(pr_w[33], 3), round(pr_u[33], 3)),
    "degree · weighted degree of node 33": (G.degree(33), G.degree(33, weight="weight")),
    "avg. clustering (count node 11 as 0 · skip it)": (round(avg_all, 3), round(avg_skip, 3)),
    "connected components": nx.number_connected_components(G),
}
pd.Series(ref, name="Python (Chapter 4)").to_frame()

,Python (Chapter 4)
average degree,4.588
density,0.139
diameter · avg. path length,"(5, 2.41)"
betweenness of node 0 (normalised · raw),"(0.438, 231.07)"
PageRank of node 33 (weights · no weights),"(0.097, 0.101)"
degree · weighted degree of node 33,"(17, 48)"
avg. clustering (count node 11 as 0 · skip it),"(0.571, 0.588)"
connected components,1


**Your note (Challenge 1):** _which number differed first, and which setting fixed it?_

## Challenge 2 · Encode your EV network — in Gephi
Open `ev_mentions_ch6.gexf`. Layout: **ForceAtlas 2** (then *Prevent overlap*). Then:
1. **Colour** → *Partition* → `community_nx`. Name each community from the table above (main group, mean sentiment) — never from the colour.
2. **Colour** → *Ranking* → `mean_sentiment` with a **diverging** palette (red – grey – blue). Which community is most negative?
3. **Size** → *Ranking* → `pagerank_nx` (min 10, max 40). Who are the most central accounts — and which group are they?
4. Check your figure in greyscale: is the message still visible?

> 💡 **Hint:** a diverging palette needs a meaningful middle — set the middle colour at 0, not at the median.

**Your note (Challenge 2):** _one finding in one sentence — with a number from the table above._

## Challenge 3 · Replay a broadcast and a viral cascade — in Gephi
Open `ev_cascade_c01.gexf`, then `ev_cascade_c05.gexf` (each in a new workspace). **Window → Timeline → Enable**, drag the window, press **Play**. Colour by `sentiment` (Partition), size by `depth` (Ranking).
Compare: how long does each cascade take, how deep does it go — and does sentiment get more negative with depth?
Export **one SVG** (Preview → *Default Straight* → Refresh → Export) with a complete caption.

> 💡 **Hint:** a complete caption names the nodes and edges, the layout and its settings, what colour means, what size means, and which labels you show.

## Hand in (with your project proposal)
One ZIP per group via Moodle: this notebook, the filled worksheet, **one SVG figure + caption**.

**Figure check:** ☐ layout named ☐ legend ☐ ≤ 8 colours ☐ caption with all settings ☐ SVG exported ☐ no user names